# Imports

In [1]:
import sys
sys.path.append('.')
import samna
import numpy as np
import matplotlib

matplotlib.use("TkAgg")          # Or "Qt5Agg", "MacOSX", "WebAgg"
import matplotlib.pyplot as plt
import samna.dynapse1 as dyn1

import dynapse1utils as ut
from netgen import Neuron, NetworkGenerator
from params_all_cores import *
import time
import importlib
from collections import deque
import threading

# sys.path.append('../Tools')

# Connect to Dynapse

In [2]:
devices = samna.device.get_unopened_devices()
print(devices)

[device::DeviceInfo(serial_number=00000033, usb_bus_number=1, usb_device_address=4, logic_version=5, device_type_name=Dynapse1DevKit)]


In [3]:
#devices = samna.device.get_unopened_devices()
model   = samna.device.open_device(devices[int(0)])

# Stimulus definition and population setup

In [4]:
api = model.get_dynapse1_api()
config1 = model.get_configuration()
param_group_c0 = config1.chips[0].cores[0].parameter_group 
param_group_c1 = config1.chips[0].cores[1].parameter_group 
param_group_c2 = config1.chips[0].cores[2].parameter_group 
param_group_c3 = config1.chips[0].cores[3].parameter_group 

In [5]:
param_list = ["IF_AHTAU_N", "IF_AHTHR_N", "IF_AHW_P", "IF_BUF_P", "IF_DC_P", "IF_NMDA_N", "IF_RFR_N", "IF_TAU1_N", "IF_TAU2_N", "IF_THR_N", "NPDPIE_TAU_F_P", "NPDPIE_TAU_S_P", "NPDPIE_THR_F_P", "NPDPIE_THR_S_P", 
              "NPDPII_TAU_F_P", "NPDPII_TAU_S_P", "NPDPII_THR_F_P", "NPDPII_THR_S_P", "PS_WEIGHT_EXC_F_N", "PS_WEIGHT_EXC_S_N", "PS_WEIGHT_INH_F_N", "PS_WEIGHT_INH_S_N", "PULSE_PWLK_P", "R2R_P"]

In [6]:
# ----------------  stimulus: a Gaussian bump ----------------
n_pts     = 1000                 # number of samples
t_end     = 1.0                  # seconds  (→ dt = 1 ms)
t         = np.linspace(0, t_end, n_pts, endpoint=False)
x         = np.linspace(-4, 4, n_pts)
sigma = 0.6  # Try smaller values: 1.0 (default), 0.5, 0.25, etc.
gauss = (1/(sigma * np.sqrt(2*np.pi))) * np.exp(-0.5 * (x / sigma)**2)
#I_peak    = 30000000e-12              # 1000 pA
I_peak    = 30000000e-12              # 1000 pA

I         = gauss/gauss.max() * I_peak   # injected current (A)

# convert to pA for nicer y‑axis numbers
I_pA = I * 1e12                 # A → pA

# ---------------- Plot stimulus waveform ----------------
plt.figure()
plt.plot(t*1e3, I_pA)           # x‑axis in ms
plt.xlabel('Time (ms)')
plt.ylabel('Injected current (pA)')
plt.title('Gaussian current stimulus (σ = 0.6)')
plt.tight_layout()

# ---------------- Plot spike times ----------------
# reproduce spike detection (same loop as user)
tau_m, R_m = 20e-3, 100e6
C_m = tau_m / R_m
v_rest = v_reset = -65e-3
v_thresh = -50e-3
t_ref  = 2e-3
dt     = t_end / n_pts


# ----------------  LIF neuron parameters ----------------------
tau_m     = 20e-3                # 20 ms membrane time constant
R_m       = 100e6                # 100 MΩ  (=> C = tau/R)
C_m       = tau_m / R_m
v_rest    = -65e-3               # -65 mV
v_reset   = -65e-3
v_thresh  = -50e-3               # spike threshold
t_ref     = 2e-3                 # 2 ms refractory period
dt        = t_end / n_pts        # simulation time-step (s)

# ----------------  simulation loop ----------------------------
v        = v_rest
next_ok  = 0.0                   # time when refractory ends
v_trace  = np.empty(n_pts)
spikes   = []

for k in range(n_pts):
    if t[k] >= next_ok:          # not in refractory
        dv = (-(v - v_rest) + R_m * I[k]) / (R_m * C_m) * dt
        v += dv
        if v >= v_thresh:        # spike!
            spikes.append(t[k])
            v = v_reset
            next_ok = t[k] + t_ref
    v_trace[k] = v

spike_times_all = np.array(spikes)

spike_ids = np.full(len(spikes), 1)

spikegen_ids = [(0, 1, n) for n in range(10)]

# separate figure for raster‑like spike markers
plt.figure()
plt.eventplot(spike_times_all*1e3, orientation='horizontal', linelength=0.1)
plt.xlabel('Time (ms)')
plt.yticks([])
plt.title(f'Spike times generated by the stimulus ({len(spike_times_all)} spikes)')
plt.tight_layout()

#plt.show()

In [7]:
eventsBuffer = deque(maxlen=500)

In [8]:
def collect_spikes(sink_node, runningFlag):
    while runningFlag[0]:  # Check first element of list
        eventsBuffer.extend(sink_node.get_events())

In [9]:
import params_all_cores

importlib.reload(params_all_cores)
config1 = model.get_configuration()

pop_nr = 4

# 1)  Declare the set of bad neurons once, in (chip, core, neuron_id) format

BROKEN_NEURONS = {(0, 1, 51), (0, 1, 71), (0, 1, 80), (0, 1, 88), (0, 1, 91)}          #  ⬅️  add more here if needed

def is_ok(chip: int, core: int, nid: int) -> bool:
    """True if this physical neuron should be used."""
    return (chip, core, nid) not in BROKEN_NEURONS

In [10]:
spikegen_offset = 2

In [11]:
import importlib, dynapse1utils as ut

importlib.reload(params_all_cores)

importlib.reload(ut)

config1 = model.get_configuration()
param_group_c0 = config1.chips[0].cores[0].parameter_group 

p_E_E   = 1
p_mexican = 1
p_I_I   = 0 #1
p_E_I   = 0 #0.1
p_I_E   = 0 #0.4

# initiate network 
net_gen = NetworkGenerator()
net_gen.clear_network()

# create spikegens, one per ring attractor neural pop 
spikegen_ids = [(0, 0, n+spikegen_offset) for n in range(10)]
#isi_spikegen = Neuron(0, 0, 200, True)
#isi_neuron = Neuron(0, 1, 200)

spikegens = []
for spikegen_id in spikegen_ids:
    spikegens.append(Neuron(spikegen_id[0], spikegen_id[1], spikegen_id[2], True))

print(spikegens)

#spikegens.append(isi_spikegen)

# Create ring neuron populations
chip = 0
core = 1
npop = 4
NBINS = 10
offset_nr = 48

ring_pops = []
next_id = offset_nr
for _ in range(NBINS):
    pop = []
    while len(pop) < npop:
        if is_ok(chip, core, next_id):
            pop.append(Neuron(chip, core, next_id))
        next_id += 1
    ring_pops.append(pop)
    
# Swap ring_pops[0] and ring_pops[1]
#ring_pops[0], ring_pops[5] = ring_pops[5], ring_pops[0]

# create inhibitory population that connects to all other pops
core_inh = 2
start_inh_neuron = 4
npop_inh = 4
pop_inhibitory = [Neuron(chip, core_inh, j) for j in range(start_inh_neuron, start_inh_neuron + npop_inh, 1)]
pop_inhibitory

# SPIKEGEN CONNECTIONS: one spike-gen (index i) permanently drives ring_pops[i]
for sg, pop in zip(spikegens, ring_pops):
    for neuron in pop:
        net_gen.add_connection(sg, neuron, dyn1.Dynapse1SynType.AMPA)
    
# connect isi spikegen to isi neuron 
#net_gen.add_connection(isi_spikegen, isi_neuron, dyn1.Dynapse1SynType.AMPA)

# self excitation in each neural population in the ring: (todo determine if this is needed) 
for pop in ring_pops:
    for pre in pop:
        for post in pop:
            if pre is not post and np.random.rand() < p_E_E:
                net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.AMPA)
                                
# MEXICAN HAT CONNECTIONS
OFFSET_1 = (-1, 1)         
for i, pop_i in enumerate(ring_pops):
    for pre in pop_i:
        for d in OFFSET_1:
            j = (i + d) % NBINS          # wrap around
            for post in ring_pops[j]:
                net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
                net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)

OFFSET_2 = (-2, 2)          # ±3 bins wide “hat”

    # todo excitatory connections to second neighbors

for i, pop_i in enumerate(ring_pops):
    for pre in pop_i:
        for d in OFFSET_2:
            j = (i + d) % NBINS          # wrap around
            for post in ring_pops[j]:
                net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)


    #  excitatory connections to third neighbors
OFFSET_3 = (-3, 3)          # ±3 bins wide “hat”
for i, pop_i in enumerate(ring_pops):
    for pre in pop_i:
        for d in OFFSET_3:
            j = (i + d) % NBINS          # wrap around
            for post in ring_pops[j]:
                #net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
                print(post)

# todo inhibitory connections to all of the other pops
OFFSET_inh = (-6, -5, -4, 4, 5, 6)          # all of the pops that are not being excited
for i, pop_i in enumerate(ring_pops):
    print(i,pop_i)
    for pre in pop_i:
        for d in OFFSET_inh:
            j = (i + d) % NBINS    
            print("j", j)# wrap around
            for post in ring_pops[j]:
                net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.GABA_B)
                #net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.GABA_B)

# INH → EXC  (global inhibition pop to all pops in the ring)
for inh in pop_inhibitory:
    for pop in ring_pops:
        for exc in pop:
            net_gen.add_connection(inh, exc, dyn1.Dynapse1SynType.GABA_B)

# EXC → INH  (drive the global inhibition pop from all pops in the ring)
for pop in ring_pops:
    for exc in pop:
        for inh in pop_inhibitory:
            net_gen.add_connection(exc, inh, dyn1.Dynapse1SynType.NMDA)

# make a dynapse1config using the network
new_config = net_gen.make_dynapse1_configuration()

# apply the configuration
model.apply_configuration(new_config)

print(net_gen.network)

# Set hardware parameters
set_params(model)

fpga_spike_gen = model.get_fpga_spike_gen() # set FPGA 


monitored_neurons = [
    (n.chip_id, n.core_id, n.neuron_id)
    for pop in ring_pops
    for n   in pop
]

monitored_neurons.extend([
    (neuron.chip_id, neuron.core_id, neuron.neuron_id)
    for neuron in pop_inhibitory  
])


graph, filter_node, sink_node = ut.create_neuron_select_graph(model, monitored_neurons)
graph.start()

# clear the buffer
sink_node.get_events()

# select the neurons to monitor
filter_node.set_neurons(monitored_neurons)

api.reset_timestamp()

ut.set_neuron_tau1(model, 0, 0, (7, 255))
ut.set_neuron_tau1(model, 0, 1, (7, 255))
ut.set_neuron_tau1(model, 0, 2, (7, 255))
ut.set_neuron_tau1(model, 0, 3, (7, 255))

time.sleep(1)

ut.set_neuron_tau1(model, 0, 0, (4, 50))
ut.set_neuron_tau1(model, 0, 1, (4, 50))
ut.set_neuron_tau1(model, 0, 2, (4, 50))
ut.set_neuron_tau1(model, 0, 3, (4, 50))


spike_ids_all = spike_ids

# Sort input events in time order
sort_indices = np.argsort(spike_times_all)
all_spike_times = spike_times_all[sort_indices]
all_spike_ids = spike_ids_all[sort_indices]

current_pop = {'value': 5}        # start with pop 0
spike_ids   = np.full(len(all_spike_times), current_pop['value'])

ut.set_fpga_spike_gen(
    fpga_spike_gen,
    all_spike_times,
    #all_spike_ids,
    spike_ids,
    #target_chips=[0] * len(all_spike_ids),
    target_chips=[0] * len(spike_ids),
    isi_base=900,
    repeat_mode=False)

fpga_spike_gen.start()

[C0c0s2, C0c0s3, C0c0s4, C0c0s5, C0c0s6, C0c0s7, C0c0s8, C0c0s9, C0c0s10, C0c0s11]
C0c1n78
C0c1n79
C0c1n81
C0c1n82
C0c1n61
C0c1n62
C0c1n63
C0c1n64
C0c1n78
C0c1n79
C0c1n81
C0c1n82
C0c1n61
C0c1n62
C0c1n63
C0c1n64
C0c1n78
C0c1n79
C0c1n81
C0c1n82
C0c1n61
C0c1n62
C0c1n63
C0c1n64
C0c1n78
C0c1n79
C0c1n81
C0c1n82
C0c1n61
C0c1n62
C0c1n63
C0c1n64
C0c1n83
C0c1n84
C0c1n85
C0c1n86
C0c1n65
C0c1n66
C0c1n67
C0c1n68
C0c1n83
C0c1n84
C0c1n85
C0c1n86
C0c1n65
C0c1n66
C0c1n67
C0c1n68
C0c1n83
C0c1n84
C0c1n85
C0c1n86
C0c1n65
C0c1n66
C0c1n67
C0c1n68
C0c1n83
C0c1n84
C0c1n85
C0c1n86
C0c1n65
C0c1n66
C0c1n67
C0c1n68
C0c1n87
C0c1n89
C0c1n90
C0c1n92
C0c1n69
C0c1n70
C0c1n72
C0c1n73
C0c1n87
C0c1n89
C0c1n90
C0c1n92
C0c1n69
C0c1n70
C0c1n72
C0c1n73
C0c1n87
C0c1n89
C0c1n90
C0c1n92
C0c1n69
C0c1n70
C0c1n72
C0c1n73
C0c1n87
C0c1n89
C0c1n90
C0c1n92
C0c1n69
C0c1n70
C0c1n72
C0c1n73
C0c1n48
C0c1n49
C0c1n50
C0c1n52
C0c1n74
C0c1n75
C0c1n76
C0c1n77
C0c1n48
C0c1n49
C0c1n50
C0c1n52
C0c1n74
C0c1n75
C0c1n76
C0c1n77
C0c1n48
C0c1n49
C0c1n

In [12]:
print(ring_pops[0], ring_pops[5])


[C0c1n48, C0c1n49, C0c1n50, C0c1n52] [C0c1n69, C0c1n70, C0c1n72, C0c1n73]


In [ ]:
ring_pops


[[C0c1n48, C0c1n49, C0c1n50, C0c1n52],
 [C0c1n53, C0c1n54, C0c1n55, C0c1n56],
 [C0c1n57, C0c1n58, C0c1n59, C0c1n60],
 [C0c1n61, C0c1n62, C0c1n63, C0c1n64],
 [C0c1n65, C0c1n66, C0c1n67, C0c1n68],
 [C0c1n69, C0c1n70, C0c1n72, C0c1n73],
 [C0c1n74, C0c1n75, C0c1n76, C0c1n77],
 [C0c1n78, C0c1n79, C0c1n81, C0c1n82],
 [C0c1n83, C0c1n84, C0c1n85, C0c1n86],
 [C0c1n87, C0c1n89, C0c1n90, C0c1n92]]

2025-08-22 11:58:48.959 python[30347:24577860] +[IMKClient subclass]: chose IMKClient_Modern
2025-08-22 11:58:48.959 python[30347:24577860] +[IMKInputSession subclass]: chose IMKInputSession_Modern


# Interactive plotting

In [14]:
# ────────────────────────────────────── #
# ======== KDE-style smoothing =========
# ────────────────────────────────────── #
import scipy.ndimage as ndi

def smooth_rates_circular(counts, sigma_bins=1.0):
    """
    Gaussian-smooth an array living on a circular ring.
    counts      : 1-D numpy array of length NBINS
    sigma_bins  : std-dev of the Gaussian, expressed in *bin* units
    """
    # Pad three bins on each side so the wrap-around is seamless
    padded = np.r_[counts[-3:], counts, counts[:3]]
    smoothed = ndi.gaussian_filter1d(padded, sigma=sigma_bins, mode='wrap')
    return smoothed[3:-3]


In [15]:
import scipy.ndimage as ndi
def smooth_rates_circular(counts, sigma_bins=1.0):
    padded = np.r_[counts[-3:], counts, counts[:3]]
    smoothed = ndi.gaussian_filter1d(padded, sigma=sigma_bins, mode='wrap')

    return smoothed[3:-3]

In [16]:
# Start spike collection in a thread
running_flag = [True]  # Use list for mutable flag
spike_thread = threading.Thread(target=collect_spikes, args=(sink_node, running_flag))
spike_thread.daemon = True
spike_thread.start()
# animation = start_spike_visualization(eventsBuffer)

In [18]:
def on_key(event):
    k = event.key
    if k.isdigit():                       # 0-9 → choose new population
        new = int(k)
        if 0 <= new < NBINS:
            current_pop['value'] = new
            fpga_spike_gen.stop()         # reload the stimuli for that pop
            spike_ids[:] = new + spikegen_offset            # update *in-place* so length stays the same
            
            """ut.set_neuron_tau1(model, 0, 0, (7, 255))
            ut.set_neuron_tau1(model, 0, 1, (7, 255))
            ut.set_neuron_tau1(model, 0, 2, (7, 255))
            ut.set_neuron_tau1(model, 0, 3, (7, 255))

            time.sleep(1)

            ut.set_neuron_tau1(model, 0, 0, (4, 50))
            ut.set_neuron_tau1(model, 0, 1, (4, 50))
            ut.set_neuron_tau1(model, 0, 2, (4, 50))
            ut.set_neuron_tau1(model, 0, 3, (4, 50))"""
            
            ut.set_fpga_spike_gen(fpga_spike_gen,
                                   all_spike_times,
                                   spike_ids,
                                   target_chips=[0]*len(spike_ids),
                                   isi_base=900,
                                   repeat_mode=False)
            fpga_spike_gen.start()
            print(f"→ Stimulating pop {new}")
    elif k == 'v':
        for i, pop_i in enumerate(ring_pops):
            for pre in pop_i:
                    j = (i + 1) % NBINS          # wrap around
                    for post in ring_pops[j]:
                        net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
        config = net_gen.make_dynapse1_configuration()
        model.apply_configuration(config)
        
    elif k == 'h':
        for i, pop_i in enumerate(ring_pops):
            for pre in pop_i:
                    j = (i + 1) % NBINS          # wrap around
                    for post in ring_pops[j]:
                        net_gen.remove_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
        config = net_gen.make_dynapse1_configuration()
        model.apply_configuration(config)
        
    elif k == 'q':                        # quit cleanly
        running_flag[0] = False

import matplotlib.colors as mcolors

# Create a mapping from neuron ID to ring population index and color
def create_neuron_to_pop_mapping(ring_pops):
    """Create mapping from neuron ID to population index"""
    neuron_to_pop = {}
    for pop_idx, pop in enumerate(ring_pops):
        for neuron in pop:
            neuron_to_pop[neuron.neuron_id] = pop_idx
    return neuron_to_pop

# Create color map for populations
colors = plt.cm.tab10(np.linspace(0, 1, NBINS))  # Use tab10 colormap for distinct colors
neuron_to_pop = create_neuron_to_pop_mapping(ring_pops)

# Set up interactive plotting
plt.ion()

# Create figure with two subplots side by side
fig, (ax_raster, ax_rate) = plt.subplots(1, 2, figsize=(15, 6))
fig.canvas.mpl_connect('key_press_event', on_key)   # moved ↓ here
fig.show()                      # ← opens ONE browser tab

# Setup raster plot in first subplot
#scatter = ax_raster.scatter([], [], s=10, alpha=0.6)
xlim_max = 10
ax_raster.set_xlabel('Time (s)')
ax_raster.set_ylabel('Neuron Index')
ax_raster.set_title('Dynap-SE1 Ring Attractor Spikes')

# Setup firing rate profile in second subplot
# Create positions for neurons (0 to 2π for the ring)
positions = np.linspace(0, 2*np.pi, NBINS, endpoint=False)

# dashed guide lines at every population centre
for p in positions:
    ax_rate.axvline(p,
                    linestyle='--',
                    linewidth=0.8,
                    color='gray',
                    alpha=0.4)

# optional: label each line with the pop index
for idx, p in enumerate(positions):
    ax_rate.text(p,            ax_rate.get_ylim()[1]*1.02,
                 str(idx),
                 ha='center', va='bottom', fontsize=9)

rate_line, = ax_rate.plot(positions, np.zeros_like(positions), 'o-', markersize=8)
ax_rate.set_xlim(0, 2*np.pi)
ax_rate.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
ax_rate.set_xticklabels(['0', 'π/2', 'π', '3π/2', '2π'])
ax_rate.set_xlabel('Position (radians)')
ax_rate.set_ylabel('Firing rate (Hz)')
ax_rate.set_title('Firing Rate Profile')
ax_rate.grid(True, alpha=0.3)

plt.tight_layout()

# Variables for rate calculation
window_size = 1.0  # seconds - time window for calculating rates
last_update_time = 0

# Real-time plotting loop
while running_flag[0]:
    try:
        if len(eventsBuffer) > 0:
            # Extract spike data for raster plot
            spikesID = [e.neuron_id for e in eventsBuffer]
            spikesTimes = [e.timestamp*1e-6 for e in eventsBuffer]  # Convert to seconds
            
            print(spikesTimes)
            print(spikesID)
            
            if not spikesID:
                continue
                
            # Update raster plot
            """ax_raster.set_ylim(min(spikesID) - 0.5, max(spikesID) + 0.5)
            scatter.set_offsets(np.column_stack((spikesTimes, spikesID)))"""
            
            # Clear previous raster plot
            ax_raster.clear()
            ax_raster.set_xlabel('Time (s)')
            ax_raster.set_ylabel('Neuron Index')
            ax_raster.set_title('Dynap-SE1 Ring Attractor Spikes')

            # Group spikes by population and plot with different colors
            for pop_idx in range(NBINS):
                # Find spikes belonging to this population
                pop_spike_times = []
                pop_spike_ids = []
                
                for spike_time, spike_id in zip(spikesTimes, spikesID):
                    if spike_id in neuron_to_pop and neuron_to_pop[spike_id] == pop_idx:
                        pop_spike_times.append(spike_time)
                        pop_spike_ids.append(spike_id)
                
                # Plot spikes for this population with its assigned color
                if pop_spike_times:  # Only plot if there are spikes
                    ax_raster.scatter(pop_spike_times, pop_spike_ids, 
                                    s=10, alpha=0.8, 
                                    color=colors[pop_idx], 
                                    label=f'Pop {pop_idx}')

            # Add legend (optional - you can remove if it clutters)
            ax_raster.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
            
            # Update time window
            current_time = max(spikesTimes)
            ax_raster.set_xlim(max(0, current_time - xlim_max), current_time + 1)
            ax_raster.set_ylim(0, 95)
            
            # Calculate firing rates across the ring (using recent time window)
            window_start = current_time - window_size
            recent_events = [e for e in eventsBuffer if e.timestamp*1e-6 >= window_start]
            
            # Count spikes for each bin in the ring
            firing_rates = np.zeros(NBINS)
            
            # Map neuron IDs to their bin/position in the ring
            """for event in recent_events:
                neuron_id = event.neuron_id
                for i, pop in enumerate(ring_pops):
                    pop_ids = [n.neuron_id for n in pop]
                    if neuron_id in pop_ids:
                        firing_rates[i] += 1
                        break"""
            # Map neuron IDs to their bin/position in the ring
            for event in recent_events:
                neuron_id = event.neuron_id
                if neuron_id in neuron_to_pop:
                    pop_idx = neuron_to_pop[neuron_id]
                    firing_rates[pop_idx] += 1
                    
            # numpy histogram for firing rate then divide by bins
            
            # Convert to Hz (spikes per second)
            firing_rates = firing_rates / window_size
            
            # Update firing rate plot
            smoothed_rates = smooth_rates_circular(firing_rates, sigma_bins=1.0)
            rate_line.set_ydata(smoothed_rates)
            max_rate = max(smoothed_rates) if any(smoothed_rates > 0) else 10
            
            ax_rate.set_ylim(0, 40)  

            
            # Refresh both plots
            fig.canvas.draw_idle()
            fig.canvas.flush_events()   # keeps the websocket alive
            time.sleep(0.01)            # tiny CPU-friendly sleep

            #plt.pause(0.0001)  # Shorter pause for smoother updates
            
    except Exception as e:
        print(f"Plotting error: {e}")
        import traceback
        traceback.print_exc()  # Print detailed error information
        break

/var/folders/38/x2v4gv396nz13mws24nz37sw0000gn/T/ipykernel_36122/1043674055.py:108: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


[1.300791, 1.389997, 1.421682, 1.491638, 1.5245229999999999, 1.6545159999999999, 1.6947189999999999, 1.738745, 1.749835, 1.76288, 1.784009, 1.793226, 1.805269, 1.8194359999999998, 1.8371579999999998, 1.901335, 1.969212, 1.9845849999999998, 1.991595, 2.0232829999999997, 2.067654, 2.069954, 2.100949, 2.118495, 2.151401, 2.209856, 2.217991, 2.228523, 2.234787, 2.244751, 2.266133, 2.2735659999999998, 2.2880059999999998, 2.345249, 2.366608, 2.3702609999999997, 2.392818, 2.408696, 2.4234709999999997, 2.473137, 2.476288, 2.494083, 2.5053389999999998, 2.506231, 2.558795, 2.57725, 2.597912, 2.655522, 2.73232, 2.7446669999999997, 2.83676, 2.838685, 2.892479, 2.89335, 2.96913, 2.989916, 3.002671, 3.0205859999999998, 3.033095, 3.0621069999999997, 3.1470089999999997, 3.1562959999999998, 3.171474, 3.1766959999999997, 3.208135, 3.2286539999999997, 3.251319, 3.3739909999999997, 3.3826959999999997, 3.5202269999999998, 3.525845, 3.56996, 3.582191, 3.6526189999999996, 3.668237, 3.673289, 3.70026599999999

In [19]:
# Create a colormap for the ring populations
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

def on_key(event):
    k = event.key
    if k.isdigit():                       # 0-9 → choose new population
        new = int(k)
        if 0 <= new < NBINS:
            current_pop['value'] = new
            fpga_spike_gen.stop()         # reload the stimuli for that pop
            spike_ids[:] = new            # update *in-place* so length stays the same
            
            """ut.set_neuron_tau1(model, 0, 0, (7, 255))
            ut.set_neuron_tau1(model, 0, 1, (7, 255))
            ut.set_neuron_tau1(model, 0, 2, (7, 255))
            ut.set_neuron_tau1(model, 0, 3, (7, 255))

            time.sleep(1)

            ut.set_neuron_tau1(model, 0, 0, (4, 50))
            ut.set_neuron_tau1(model, 0, 1, (4, 50))
            ut.set_neuron_tau1(model, 0, 2, (4, 50))
            ut.set_neuron_tau1(model, 0, 3, (4, 50))"""
            
            ut.set_fpga_spike_gen(fpga_spike_gen,
                                   all_spike_times,
                                   spike_ids,
                                   target_chips=[0]*len(spike_ids),
                                   isi_base=900,
                                   repeat_mode=False)
            fpga_spike_gen.start()
            print(f"→ Stimulating pop {new}")
    elif k == 'v':
        for i, pop_i in enumerate(ring_pops):
            for pre in pop_i:
                    j = (i + 1) % NBINS          # wrap around
                    for post in ring_pops[j]:
                        net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
        config = net_gen.make_dynapse1_configuration()
        model.apply_configuration(config)
        
    elif k == 'h':
        for i, pop_i in enumerate(ring_pops):
            for pre in pop_i:
                    j = (i + 1) % NBINS          # wrap around
                    for post in ring_pops[j]:
                        net_gen.remove_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
        config = net_gen.make_dynapse1_configuration()
        model.apply_configuration(config)
        
    elif k == 'q':                        # quit cleanly
        running_flag[0] = False


# Create a mapping from neuron ID to ring population index and color
def create_neuron_to_pop_mapping(ring_pops):
    """Create mapping from neuron ID to population index"""
    neuron_to_pop = {}
    for pop_idx, pop in enumerate(ring_pops):
        for neuron in pop:
            neuron_to_pop[neuron.neuron_id] = pop_idx
    return neuron_to_pop

# Create color map for populations
colors = plt.cm.tab10(np.linspace(0, 1, NBINS))  # Use tab10 colormap for distinct colors
neuron_to_pop = create_neuron_to_pop_mapping(ring_pops)

# Set up interactive plotting
plt.ion()

# Create figure with two subplots side by side
fig, (ax_raster, ax_rate) = plt.subplots(1, 2, figsize=(15, 6))
fig.canvas.mpl_connect('key_press_event', on_key)
fig.show()

# Setup raster plot in first subplot - Remove the old scatter plot
ax_raster.set_xlabel('Time (s)')
ax_raster.set_ylabel('Neuron Index')
ax_raster.set_title('Dynap-SE1 Ring Attractor Spikes')

# Setup firing rate profile in second subplot
positions = np.linspace(0, 2*np.pi, NBINS, endpoint=False)

# dashed guide lines at every population centre
for p in positions:
    ax_rate.axvline(p,
                    linestyle='--',
                    linewidth=0.8,
                    color='gray',
                    alpha=0.4)

# optional: label each line with the pop index
for idx, p in enumerate(positions):
    ax_rate.text(p, ax_rate.get_ylim()[1]*1.02,
                 str(idx),
                 ha='center', va='bottom', fontsize=9)

rate_line, = ax_rate.plot(positions, np.zeros_like(positions), 'o-', markersize=8)
ax_rate.set_xlim(0, 2*np.pi)
ax_rate.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
ax_rate.set_xticklabels(['0', 'π/2', 'π', '3π/2', '2π'])
ax_rate.set_xlabel('Position (radians)')
ax_rate.set_ylabel('Firing rate (Hz)')
ax_rate.set_title('Firing Rate Profile')
ax_rate.grid(True, alpha=0.3)

plt.tight_layout()

# Variables for rate calculation
window_size = 1.0
last_update_time = 0

# Real-time plotting loop
while running_flag[0]:
    try:
        if len(eventsBuffer) > 0:
            # Extract spike data for raster plot
            spikesID = [e.neuron_id for e in eventsBuffer]
            spikesTimes = [e.timestamp*1e-6 for e in eventsBuffer]
            
            print(spikesTimes)
            print(spikesID)
            
            if not spikesID:
                continue
            
            # Clear previous raster plot
            ax_raster.clear()
            ax_raster.set_xlabel('Time (s)')
            ax_raster.set_ylabel('Neuron Index')
            ax_raster.set_title('Dynap-SE1 Ring Attractor Spikes')
            
            # Group spikes by population and plot with different colors
            for pop_idx in range(NBINS):
                # Find spikes belonging to this population
                pop_spike_times = []
                pop_spike_ids = []
                
                for spike_time, spike_id in zip(spikesTimes, spikesID):
                    if spike_id in neuron_to_pop and neuron_to_pop[spike_id] == pop_idx:
                        pop_spike_times.append(spike_time)
                        pop_spike_ids.append(spike_id)
                
                # Plot spikes for this population with its assigned color
                if pop_spike_times:  # Only plot if there are spikes
                    ax_raster.scatter(pop_spike_times, pop_spike_ids, 
                                    s=10, alpha=0.8, 
                                    color=colors[pop_idx], 
                                    label=f'Pop {pop_idx}')
            
            # Update time window
            current_time = max(spikesTimes)
            ax_raster.set_xlim(max(0, current_time - xlim_max), current_time + 1)
            ax_raster.set_ylim(0, 95)
            
            # Add legend (optional - you can remove if it clutters)
            ax_raster.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
            
            # Calculate firing rates across the ring (using recent time window)
            window_start = current_time - window_size
            recent_events = [e for e in eventsBuffer if e.timestamp*1e-6 >= window_start]
            
            # Count spikes for each bin in the ring
            firing_rates = np.zeros(NBINS)
            
            # Map neuron IDs to their bin/position in the ring
            for event in recent_events:
                neuron_id = event.neuron_id
                if neuron_id in neuron_to_pop:
                    pop_idx = neuron_to_pop[neuron_id]
                    firing_rates[pop_idx] += 1
            
            # Convert to Hz (spikes per second)
            firing_rates = firing_rates / window_size
            
            # Update firing rate plot
            smoothed_rates = smooth_rates_circular(firing_rates, sigma_bins=1.0)
            rate_line.set_ydata(smoothed_rates)
            max_rate = max(smoothed_rates) if any(smoothed_rates > 0) else 10
            
            ax_rate.set_ylim(0, 40)
            
            # Refresh both plots
            fig.canvas.draw_idle()
            fig.canvas.flush_events()
            time.sleep(0.01)
            
    except Exception as e:
        print(f"Plotting error: {e}")
        import traceback
        traceback.print_exc()
        break

[22.897956999999998, 22.913856, 22.924360999999998, 23.091918999999997, 23.096731, 23.106911, 23.119248, 23.169173, 23.18209, 23.200848, 23.22955, 23.234574, 23.273062, 23.293135, 23.295704, 23.301887, 23.395063999999998, 23.401536, 23.452025, 23.472361, 23.484227, 23.523502999999998, 23.527931, 23.536932, 23.585378, 23.671436, 23.714472999999998, 23.715889, 23.721103, 23.818073, 23.82761, 23.870635999999998, 23.888855, 23.891135, 23.969631, 24.001542999999998, 24.012435, 24.026815, 24.034354999999998, 24.110035, 24.144185999999998, 24.157732, 24.179164, 24.201722, 24.214724, 24.228400999999998, 24.263216999999997, 24.333343, 24.372562, 24.40945, 24.454756, 24.491854999999997, 24.516631999999998, 24.524251, 24.540307, 24.552173, 24.556745, 24.618664, 24.6343, 24.666522, 24.695940999999998, 24.708534999999998, 24.749126999999998, 24.799455, 24.821832999999998, 24.840816999999998, 24.889733999999997, 24.950219999999998, 25.002722, 25.028052, 25.03595, 25.062134, 25.078367, 25.127616, 25.

/var/folders/38/x2v4gv396nz13mws24nz37sw0000gn/T/ipykernel_32776/2292087932.py:109: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
Traceback (most recent call last):
  File "/var/folders/38/x2v4gv396nz13mws24nz37sw0000gn/T/ipykernel_32776/2292087932.py", line 155, in <module>
    ax_raster.set_xlim(max(0, current_time - xlim_max), current_time + 1)
                                             ^^^^^^^^
NameError: name 'xlim_max' is not defined


In [ ]:
# To stop everything cleanly:
running_flag[0] = False
spike_thread.join(timeout=1.0)
plt.ioff()
plt.close()

# Velocity simulation

In [14]:
# ===== OFFLINE SIMULATION WITH VELOCITY MODULATION =====

print("Starting offline simulation sequence...")

time_phase1 = 2.0
time_phase2 = 7.0
time_phase3 = 3.0

p_vel = 0.5

conn = 1

# Initialize storage for all events
all_events = []
simulation_phases = []  # Track which phase each event belongs to


ut.set_neuron_tau1(model, 0, 1, (4, 50))
ut.set_neuron_tau1(model, 0, 2, (4, 200))# save the drift for this cycle

# Clear any existing events
eventsBuffer.clear()
sink_node.get_events()

# === PHASE 1: Initial stimulation of population 5 for 2 seconds ===
print(f"Phase 1: Stimulating population 5 for {time_phase1} seconds...")

# Set FPGA to stimulate population 5
spike_ids[:] = 2+spikegen_offset  # Set all spike IDs to 5 (population 5)
ut.set_fpga_spike_gen(fpga_spike_gen,
                      all_spike_times, spike_ids,
                      target_chips=[0]*len(spike_ids),
                      isi_base=900, repeat_mode=False)

# Start simulation
fpga_spike_gen.start()
phase1_start = time.time()

while time.time() - phase1_start < time_phase1:
    new_events = sink_node.get_events()
    for event in new_events:
        all_events.append(event)
        simulation_phases.append("Phase 1: Initial")
    time.sleep(0.01)

fpga_spike_gen.stop()
print(f"Phase 1 complete. Collected {len([p for p in simulation_phases if p == 'Phase 1: Initial'])} events.")

# === PHASE 2: Add velocity connections and run for 7 seconds ===
print(f"Phase 2: Adding velocity connections and running for {time_phase2} seconds...")

pre_list = []
post_list = []

# Add velocity connections (equivalent to 'v' key press)
for i, pop_i in enumerate(ring_pops):
    for pre in pop_i:
        j = (i + 1) % NBINS  # wrap around
        for post in ring_pops[j]:
            #if pre is not post and np.random.rand() < p_vel:
            net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
                #print("pre vel neuron", pre)
                #print("post vel neuron", post)
            pre_list.append(pre)
            post_list.append(post)

# Apply new configuration
config = net_gen.make_dynapse1_configuration()
model.apply_configuration(config)
print("Velocity connections added.")

# Continue stimulation
# fpga_spike_gen.start()
phase2_start = time.time()

while time.time() - phase2_start < time_phase2:
    new_events = sink_node.get_events()
    for event in new_events:
        all_events.append(event)
        simulation_phases.append("Phase 2: Velocity")
    time.sleep(0.01)

# fpga_spike_gen.stop()
print(f"Phase 2 complete. Collected {len([p for p in simulation_phases if p == 'Phase 2: Velocity'])} events.")

# === PHASE 3: Remove velocity connections and run for 3 seconds ===
print(f"Phase 3: Removing velocity connections and running for {time_phase3} seconds...")

# Remove velocity connections (equivalent to 'h' key press)
for i, pop_i in enumerate(ring_pops):
    for pre in pop_i:
        j = (i + 1) % NBINS  # wrap around
        for post in ring_pops[j]:
            net_gen.remove_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
            
"""for i in range(len(pre_list)):
    net_gen.remove_connection(pre_list[i], post_list[i], dyn1.Dynapse1SynType.NMDA)"""

# Apply new configuration
config = net_gen.make_dynapse1_configuration()
model.apply_configuration(config)
print("Velocity connections removed.")

# Continue stimulation
# fpga_spike_gen.start()
phase3_start = time.time()

while time.time() - phase3_start < time_phase3:
    new_events = sink_node.get_events()
    for event in new_events:
        all_events.append(event)
        simulation_phases.append("Phase 3: No velocity")
    time.sleep(0.01)

# fpga_spike_gen.stop()
print(f"Phase 3 complete. Collected {len([p for p in simulation_phases if p == 'Phase 3: No velocity'])} events.")

# Final event collection
final_events = sink_node.get_events()
for event in final_events:
    all_events.append(event)
    simulation_phases.append("Phase 3: No velocity")

print(f"Simulation complete! Total events collected: {len(all_events)}")


"""# === END OF SIMULATION === #"""

"""# === PLOTTING PART === #"""

# === RASTER PLOT RESULTS ===
print("Creating raster plot...")

if len(all_events) > 0:
    # Extract event data and normalize timestamps to start from 0
    raw_timestamps = [e.timestamp * 1e-6 for e in all_events]  # Convert to seconds
    start_time = min(raw_timestamps)  # Get the first timestamp
    event_times = [t - start_time for t in raw_timestamps]  # Normalize to start from 0
    event_neuron_ids = [e.neuron_id for e in all_events]
    
    # Create neuron ID to population mapping for coloring
    nid_to_pop = {}
    for pop_idx, pop in enumerate(ring_pops):
        for n in pop:
            nid_to_pop[n.neuron_id] = pop_idx
    
    # Give inhibitory population its own index
    INH_IDX = NBINS
    for n in pop_inhibitory:
        nid_to_pop[n.neuron_id] = INH_IDX
    
    # Map colors
    import matplotlib.cm as cm
    pop_colors = cm.get_cmap('tab10', NBINS+1)
    evt_colors = []
    for nid in event_neuron_ids:
        pop_idx = nid_to_pop.get(nid, None)
        if pop_idx is None or pop_idx == INH_IDX:
            evt_colors.append('lightgrey')
        else:
            evt_colors.append(pop_colors(pop_idx))
    
    # Create the plot
    plt.figure(figsize=(15, 8))
    plt.scatter(event_times, event_neuron_ids, s=3, c=evt_colors, alpha=0.7)
    
    # Add vertical lines to mark phase transitions
    plt.axvline(time_phase1, color='red', linestyle='--', linewidth=2, alpha=0.8, label='Velocity ON')
    plt.axvline(time_phase1 + time_phase2, color='blue', linestyle='--', linewidth=2, alpha=0.8, label='Velocity OFF') 
    
    # Add population labels
    for pop in range(NBINS):
        ids = [n.neuron_id for n in ring_pops[pop]]
        y_lim =max(ids)
        # if ids:
        #     y_pos = np.mean(ids)
        #     plt.text(-0.5, y_pos, f'P{pop}', va='center', ha='right', fontsize=8, 
        #             color=pop_colors(pop), weight='bold')
    
    plt.ylim(0,y_lim+5)
    plt.xlabel('Time (s)', fontsize=12)
    plt.ylabel('Neuron ID', fontsize=12)
    plt.title(f'Offline Simulation: Population 5 Stimulation with Velocity Modulation\n' + 
              f'Phase 1 (0-{time_phase1}s): Initial | Phase 2 ({time_phase1}-{time_phase1+time_phase2}s): +Velocity | Phase 3 ({time_phase1+time_phase2}-{time_phase1+time_phase2+time_phase3}s): -Velocity', 
              fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.legend()
    
    # Print summary statistics
    phase1_events = len([p for p in simulation_phases if p == 'Phase 1: Initial'])
    phase2_events = len([p for p in simulation_phases if p == 'Phase 2: Velocity']) 
    phase3_events = len([p for p in simulation_phases if p == 'Phase 3: No velocity'])
    
    print(f"\nSimulation Summary:")
    print(f"Phase 1 (Initial, 0-{time_phase1}s): {phase1_events} events")
    print(f"Phase 2 (Velocity, {time_phase1}-{time_phase1+time_phase2}s): {phase2_events} events") 
    print(f"Phase 3 (No velocity, {time_phase1+time_phase2}-{time_phase1+time_phase2+time_phase3}s): {phase3_events} events")
    print(f"Total events: {len(all_events)}")
    print(f"Total duration: {time_phase1+time_phase2+time_phase3} seconds")
    
    plt.tight_layout()
    
    
    
    # === PEAK TRACKING ANALYSIS ===
    print("\nStarting peak tracking analysis...")
    
    from scipy.optimize import minimize
    from scipy.stats import circmean
    
    def circular_gaussian(x, mu, sigma, A):
        
        # Handle periodic boundary conditions
        diff = np.array([(xi - mu + NBINS/2) % NBINS - NBINS/2 for xi in x])
        return A * np.exp(-0.5 * (diff / sigma)**2)
    
    def fit_circular_gaussian(firing_rates):
        positions = np.arange(NBINS)
        
        # Skip if no activity
        if np.sum(firing_rates) == 0:
            return np.nan, np.nan, np.nan  # mu, sigma, A
        
        # Initial guess: peak at maximum firing rate position
        max_pos = np.argmax(firing_rates)
        initial_A = np.max(firing_rates)
        initial_sigma = 1.0
        
        def objective(params):
            mu, sigma, A = params
            if sigma <= 0 or A < 0:
                return 1e10
            predicted = circular_gaussian(positions, mu, sigma, A)
            return np.sum((firing_rates - predicted)**2)
        
        # Try optimization with different initial conditions
        best_result = None
        best_error = np.inf
        
        for init_mu in [max_pos, (max_pos + 1) % NBINS, (max_pos - 1) % NBINS]:
            try:
                result = minimize(objective, [init_mu, initial_sigma, initial_A],
                                method='L-BFGS-B',
                                bounds=[(0, NBINS-1), (0.1, NBINS/2), (0, None)])
                if result.success and result.fun < best_error:
                    best_result = result
                    best_error = result.fun
            except:
                continue
        
        if best_result is not None:
            mu, sigma, A = best_result.x
            # Normalize mu to [0, NBINS) range
            mu = mu % NBINS
            return mu, sigma, A
        else:
            return np.nan, np.nan, np.nan
    
    # Analysis parameters
    timestep = 0.5 # seconds
    total_duration = time_phase1 + time_phase2 + time_phase3  # seconds
    time_bins = np.arange(0, total_duration + timestep, timestep)
    
    # Storage for results
    peak_positions = []
    peak_times = []
    all_firing_rates = []
    fit_quality = []
    
    print(f"Analyzing {len(time_bins)-1} time windows...")
    
    for i in range(len(time_bins) - 1):
        t_start = time_bins[i]
        t_end = time_bins[i + 1]
        
        # Find events in this time window
        window_events = []
        for j, event_time in enumerate(event_times):
            if t_start <= event_time < t_end:
                window_events.append((event_time, event_neuron_ids[j]))
        
        # Calculate firing rates for each population
        firing_rates = np.zeros(NBINS)
        for event_time, neuron_id in window_events:
            if neuron_id in nid_to_pop and nid_to_pop[neuron_id] < NBINS:  # Exclude inhibitory
                pop_idx = nid_to_pop[neuron_id]
                firing_rates[pop_idx] += 1
        
        # Convert to Hz (events per second)
        firing_rates = firing_rates / timestep
        all_firing_rates.append(firing_rates.copy())
        
        # Fit circular Gaussian
        mu, sigma, A = fit_circular_gaussian(firing_rates)
        
        if not np.isnan(mu):
            peak_positions.append(mu)
            peak_times.append(t_start + timestep/2)  # Center of time window
            
            # Calculate fit quality (R-squared)
            predicted = circular_gaussian(np.arange(NBINS), mu, sigma, A)
            ss_res = np.sum((firing_rates - predicted)**2)
            ss_tot = np.sum((firing_rates - np.mean(firing_rates))**2)
            r_squared = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
            fit_quality.append(r_squared)
        else:
            fit_quality.append(0)
    
    peak_positions = np.array(peak_positions)
    peak_times = np.array(peak_times)
    fit_quality = np.array(fit_quality)
    
    # Convert peak positions to degrees (360° / 10 bins = 36° per bin)
    peak_positions_degrees = peak_positions * 360.0 / NBINS
    
    plt.figure(figsize=(12, 6))
    plt.plot(peak_times, peak_positions_degrees, 'ro', markersize=2, alpha=0.8, label='Peak Position')
    

    plt.axvline(time_phase1, color='red', linestyle='--', linewidth=2, alpha=0.8, label='Velocity ON')
    plt.axvline(time_phase1 + time_phase2, color='blue', linestyle='--', linewidth=2, alpha=0.8, label='Velocity OFF')
    
    plt.xlabel('Time (s)', fontsize=12)
    plt.ylabel('Peak Position (degrees)', fontsize=12)
    plt.title('Ring Attractor Peak Position Over Time\n(Population Activity Center Tracking)', fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=10)
    
    plt.ylim(0, 360)
    plt.yticks(np.arange(0, 361, 45), [f'{int(deg)}°' for deg in np.arange(0, 361, 45)])
    

    for deg in np.arange(0, 361, 90):
        plt.axhline(deg, color='gray', linestyle=':', alpha=0.2, linewidth=0.8)
    
    plt.tight_layout()
    plt.show()

    print(f"Successfully tracked {len(peak_positions)} peaks out of {len(time_bins)-1} time windows")  
 
else:
    print("No events collected during simulation!")
    

ut.set_neuron_tau1(model, 0, 1, (7, 255))
ut.set_neuron_tau1(model, 0, 2, (7, 255))



Starting offline simulation sequence...
Phase 1: Stimulating population 5 for 2.0 seconds...
Phase 1 complete. Collected 9 events.
Phase 2: Adding velocity connections and running for 7.0 seconds...
Velocity connections added.
Phase 2 complete. Collected 256 events.
Phase 3: Removing velocity connections and running for 3.0 seconds...
Velocity connections removed.
Phase 3 complete. Collected 103 events.
Simulation complete! Total events collected: 369
Creating raster plot...

Simulation Summary:
Phase 1 (Initial, 0-2.0s): 9 events
Phase 2 (Velocity, 2.0-9.0s): 256 events
Phase 3 (No velocity, 9.0-12.0s): 104 events
Total events: 369
Total duration: 12.0 seconds

Starting peak tracking analysis...


/var/folders/38/x2v4gv396nz13mws24nz37sw0000gn/T/ipykernel_30347/2904538267.py:154: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  pop_colors = cm.get_cmap('tab10', NBINS+1)


Analyzing 24 time windows...
Successfully tracked 24 peaks out of 24 time windows


In [15]:
# ===== REUSABLE FUNCTIONS FOR COMPLEX SIMULATIONS =====

def circular_gaussian(x, mu, sigma, A):
    
    # Handle periodic boundary conditions
    diff = np.array([(xi - mu + NBINS/2) % NBINS - NBINS/2 for xi in x])
    return A * np.exp(-0.5 * (diff / sigma)**2)

def fit_circular_gaussian(firing_rates):
    positions = np.arange(NBINS)
    
    # Skip if no activity
    if np.sum(firing_rates) == 0:
        return np.nan, np.nan, np.nan  # mu, sigma, A
    
    # Initial guess: peak at maximum firing rate position
    max_pos = np.argmax(firing_rates)
    initial_A = np.max(firing_rates)
    initial_sigma = 1.0
    
    def objective(params):
        mu, sigma, A = params
        if sigma <= 0 or A < 0:
            return 1e10
        predicted = circular_gaussian(positions, mu, sigma, A)
        return np.sum((firing_rates - predicted)**2)
    
    # Try optimization with different initial conditions
    best_result = None
    best_error = np.inf
    
    for init_mu in [max_pos, (max_pos + 1) % NBINS, (max_pos - 1) % NBINS]:
        try:
            result = minimize(objective, [init_mu, initial_sigma, initial_A],
                            method='L-BFGS-B',
                            bounds=[(0, NBINS-1), (0.1, NBINS/2), (0, None)])
            if result.success and result.fun < best_error:
                best_result = result
                best_error = result.fun
                
        except:
            continue
    
    if best_result is not None:
        mu, sigma, A = best_result.x
        # Normalize mu to [0, NBINS) range
        mu = mu % NBINS
        return mu, sigma, A
    else:
        return np.nan, np.nan, np.nan


def run_offline_simulation(time_phase1, time_phase2, time_phase3, spike_pop_id):
    """
    Run offline simulation with three phases and velocity modulation.
    
    Parameters:
    - time_phase1: Duration of initial phase (seconds)
    - time_phase2: Duration of velocity phase (seconds) 
    - time_phase3: Duration of no-velocity phase (seconds)
    - spike_pop_id: Population ID to stimulate (0-9)
    
    Returns:
    - all_events: List of collected spike events
    """
    print(f"\nStarting offline simulation: Pop {spike_pop_id}, phases: {time_phase1}s, {time_phase2}s, {time_phase3}s")
    
    # Initialize storage for all events
    all_events = []
    simulation_phases = []
    
    # Clear any existing events
    eventsBuffer.clear()
    sink_node.get_events()
    
    # === PHASE 1: Initial stimulation ===
    print(f"Phase 1: Stimulating population {spike_pop_id} for {time_phase1} seconds...")
    
    # Set FPGA to stimulate specified population
    spike_ids[:] = spike_pop_id + spikegen_offset
    ut.set_fpga_spike_gen(fpga_spike_gen,
                          all_spike_times, spike_ids,
                          target_chips=[0]*len(spike_ids),
                          isi_base=900, repeat_mode=False)
    
    # Start simulation
    api.reset_timestamp()
    fpga_spike_gen.start()
    phase1_start = time.time()
    
    while time.time() - phase1_start < time_phase1:
        new_events = sink_node.get_events()
        for event in new_events:
            all_events.append(event)
            simulation_phases.append("Phase 1: Initial")
        time.sleep(0.01)
    
    fpga_spike_gen.stop()
    print(f"Phase 1 complete. Collected {len([p for p in simulation_phases if p == 'Phase 1: Initial'])} events.")
    
    # === PHASE 2: Add velocity connections ===
    print(f"Phase 2: Adding velocity connections and running for {time_phase2} seconds...")
    
    pre_list = []
    post_list = []
    
    if conn == 1:
        for i, pop_i in enumerate(ring_pops):
            for p, pre in enumerate(pop_i):
                j = (i + 1) % NBINS
                post = ring_pops[j][p]
                net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)

    if conn == 2:
        for i, pop_i in enumerate(ring_pops):
            for p, pre in enumerate(pop_i):
                j = (i + 1) % NBINS
                post1 = ring_pops[j][p]
                post2 = ring_pops[j][(p+1)%4]
                net_gen.add_connection(pre, post1, dyn1.Dynapse1SynType.NMDA)   
                net_gen.add_connection(pre, post2, dyn1.Dynapse1SynType.NMDA)   

    if conn == 3:
        for i, pop_i in enumerate(ring_pops):
            print("i", i)
            for p, pre in enumerate(pop_i):
                j = (i + 1) % NBINS
                post1 = ring_pops[j][p]
                post2 = ring_pops[j][(p+1)%4]
                post3 = ring_pops[j][(p+2)%4]
                net_gen.add_connection(pre, post1, dyn1.Dynapse1SynType.NMDA)   
                net_gen.add_connection(pre, post2, dyn1.Dynapse1SynType.NMDA)   
                net_gen.add_connection(pre, post3, dyn1.Dynapse1SynType.NMDA)   
    
    if conn == 4:
    
        # Add velocity connections
        for i, pop_i in enumerate(ring_pops):
            for pre in pop_i:
                j = (i + 1) % NBINS
                for post in ring_pops[j]:
                    #if pre is not post and np.random.rand() < p_vel:
                    #if np.random.rand() < p_vel:
                    net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
                        #print("pre vel neuron", pre)
                        #print("post vel neuron", post)
                    pre_list.append(pre)
                    post_list.append(post)
    
    if conn == 5:
        for i, pop_i in enumerate(ring_pops):
            for pre in pop_i:
                j = (i + 1) % NBINS
                for post in ring_pops[j]:
                    #if pre is not post and np.random.rand() < p_vel:
                    #if np.random.rand() < p_vel:
                    net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
                        #print("pre vel neuron", pre)
                        #print("post vel neuron", post)
                    pre_list.append(pre)
                    post_list.append(post)        
        for i, pop_i in enumerate(ring_pops):
            for p, pre in enumerate(pop_i):
                j = (i + 1) % NBINS
                post = ring_pops[j][p]
                net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
                
    if conn == 6:     
        for i, pop_i in enumerate(ring_pops):
            for pre in pop_i:
                j = (i + 1) % NBINS
                for post in ring_pops[j]:
                    #if pre is not post and np.random.rand() < p_vel:
                    #if np.random.rand() < p_vel:
                    net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
                        #print("pre vel neuron", pre)
                        #print("post vel neuron", post)
                    pre_list.append(pre)
                    post_list.append(post)   
                    
        for i, pop_i in enumerate(ring_pops):
            for p, pre in enumerate(pop_i):
                j = (i + 1) % NBINS
                post1 = ring_pops[j][p]
                post2 = ring_pops[j][(p+1)%4]
                net_gen.add_connection(pre, post1, dyn1.Dynapse1SynType.NMDA)   
                net_gen.add_connection(pre, post2, dyn1.Dynapse1SynType.NMDA)  
                
                
    if conn == 7:
        for i, pop_i in enumerate(ring_pops):
            for pre in pop_i:
                j = (i + 1) % NBINS
                for post in ring_pops[j]:
                    #if pre is not post and np.random.rand() < p_vel:
                    #if np.random.rand() < p_vel:
                    net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
                        #print("pre vel neuron", pre)
                        #print("post vel neuron", post)
                    pre_list.append(pre)
                    post_list.append(post)
        for i, pop_i in enumerate(ring_pops):
            print("i", i)
            for p, pre in enumerate(pop_i):
                j = (i + 1) % NBINS
                post1 = ring_pops[j][p]
                post2 = ring_pops[j][(p+1)%4]
                post3 = ring_pops[j][(p+2)%4]
                net_gen.add_connection(pre, post1, dyn1.Dynapse1SynType.NMDA)   
                net_gen.add_connection(pre, post2, dyn1.Dynapse1SynType.NMDA)   
                net_gen.add_connection(pre, post3, dyn1.Dynapse1SynType.NMDA)   

    if conn == 8:
        for i, pop_i in enumerate(ring_pops):
            for pre in pop_i:
                j = (i + 1) % NBINS
                for post in ring_pops[j]:
                    #if pre is not post and np.random.rand() < p_vel:
                    #if np.random.rand() < p_vel:
                    net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
                    net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
                        #print("pre vel neuron", pre)
                        #print("post vel neuron", post)
                    pre_list.append(pre)
                    post_list.append(post)
        
        
    # Apply new configuration
    config = net_gen.make_dynapse1_configuration()
    model.apply_configuration(config)
    print("Velocity connections added.")
    
    # Continue stimulation
    phase2_start = time.time()
    
    while time.time() - phase2_start < time_phase2:
        new_events = sink_node.get_events()
        for event in new_events:
            all_events.append(event)
            simulation_phases.append("Phase 2: Velocity")
        time.sleep(0.01)
    
    print(f"Phase 2 complete. Collected {len([p for p in simulation_phases if p == 'Phase 2: Velocity'])} events.")
    
    # === PHASE 3: Remove velocity connections ===
    print(f"Phase 3: Removing velocity connections and running for {time_phase3} seconds...")

    if conn == 1:
        for i, pop_i in enumerate(ring_pops):
            for p, pre in enumerate(pop_i):
                j = (i + 1) % NBINS
                post = ring_pops[j][p]
                net_gen.remove_connection(pre, post, dyn1.Dynapse1SynType.NMDA)

    if conn == 2:
        for i, pop_i in enumerate(ring_pops):
            for p, pre in enumerate(pop_i):
                j = (i + 1) % NBINS
                post1 = ring_pops[j][p]
                post2 = ring_pops[j][(p+1)%4]
                net_gen.remove_connection(pre, post1, dyn1.Dynapse1SynType.NMDA)   
                net_gen.remove_connection(pre, post2, dyn1.Dynapse1SynType.NMDA)   
                
    if conn == 3:
        for i, pop_i in enumerate(ring_pops):
            for p, pre in enumerate(pop_i):
                j = (i + 1) % NBINS
                post1 = ring_pops[j][p]
                post2 = ring_pops[j][(p+1)%4]
                post3 = ring_pops[j][(p+2)%4]
                net_gen.remove_connection(pre, post1, dyn1.Dynapse1SynType.NMDA)   
                net_gen.remove_connection(pre, post2, dyn1.Dynapse1SynType.NMDA)   
                net_gen.remove_connection(pre, post3, dyn1.Dynapse1SynType.NMDA)   
    
    if conn == 4: 
        for i, pop_i in enumerate(ring_pops):
            for pre in pop_i:
                j = (i + 1) % NBINS
                for post in ring_pops[j]:
                    net_gen.remove_connection(pre, post, dyn1.Dynapse1SynType.NMDA)

    if conn == 5:
        for i, pop_i in enumerate(ring_pops):
            for pre in pop_i:
                j = (i + 1) % NBINS
                for post in ring_pops[j]:
                    #if pre is not post and np.random.rand() < p_vel:
                    #if np.random.rand() < p_vel:
                    net_gen.remove_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
                        #print("pre vel neuron", pre)
                        #print("post vel neuron", post)
                    pre_list.append(pre)
                    post_list.append(post)        
        for i, pop_i in enumerate(ring_pops):
            for p, pre in enumerate(pop_i):
                j = (i + 1) % NBINS
                post = ring_pops[j][p]
                net_gen.remove_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
                
    if conn == 6:     
        for i, pop_i in enumerate(ring_pops):
            for pre in pop_i:
                j = (i + 1) % NBINS
                for post in ring_pops[j]:
                    #if pre is not post and np.random.rand() < p_vel:
                    #if np.random.rand() < p_vel:
                    net_gen.remove_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
                        #print("pre vel neuron", pre)
                        #print("post vel neuron", post)
                    pre_list.append(pre)
                    post_list.append(post)   
                    
        for i, pop_i in enumerate(ring_pops):
            for p, pre in enumerate(pop_i):
                j = (i + 1) % NBINS
                post1 = ring_pops[j][p]
                post2 = ring_pops[j][(p+1)%4]
                net_gen.remove_connection(pre, post1, dyn1.Dynapse1SynType.NMDA)   
                net_gen.remove_connection(pre, post2, dyn1.Dynapse1SynType.NMDA)  

    if conn == 7:
        for i, pop_i in enumerate(ring_pops):
            for pre in pop_i:
                j = (i + 1) % NBINS
                for post in ring_pops[j]:
                    #if pre is not post and np.random.rand() < p_vel:
                    #if np.random.rand() < p_vel:
                    net_gen.remove_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
                        #print("pre vel neuron", pre)
                        #print("post vel neuron", post)
                    pre_list.append(pre)
                    post_list.append(post)
        for i, pop_i in enumerate(ring_pops):
            print("i", i)
            for p, pre in enumerate(pop_i):
                j = (i + 1) % NBINS
                post1 = ring_pops[j][p]
                post2 = ring_pops[j][(p+1)%4]
                post3 = ring_pops[j][(p+2)%4]
                net_gen.remove_connection(pre, post1, dyn1.Dynapse1SynType.NMDA)   
                net_gen.remove_connection(pre, post2, dyn1.Dynapse1SynType.NMDA)   
                net_gen.remove_connection(pre, post3, dyn1.Dynapse1SynType.NMDA)   
                
    if conn == 8:
        for i, pop_i in enumerate(ring_pops):
            for pre in pop_i:
                j = (i + 1) % NBINS
                for post in ring_pops[j]:
                    #if pre is not post and np.random.rand() < p_vel:
                    #if np.random.rand() < p_vel:
                    net_gen.remove_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
                    net_gen.remove_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
                        #print("pre vel neuron", pre)
                        #print("post vel neuron", post)
                    pre_list.append(pre)
                    post_list.append(post)
               
    """for i in range(len(pre_list)):
        net_gen.remove_connection(pre_list[i], post_list[i], dyn1.Dynapse1SynType.NMDA)"""
    

    # Apply new configuration
    config = net_gen.make_dynapse1_configuration()
    model.apply_configuration(config)
    print("Velocity connections removed.")
    
    # Continue stimulation
    phase3_start = time.time()
    
    while time.time() - phase3_start < time_phase3:
        new_events = sink_node.get_events()
        for event in new_events:
            all_events.append(event)
            simulation_phases.append("Phase 3: No velocity")
        time.sleep(0.01)
    
    print(f"Phase 3 complete. Collected {len([p for p in simulation_phases if p == 'Phase 3: No velocity'])} events.")
    
    # Final event collection
    final_events = sink_node.get_events()
    for event in final_events:
        all_events.append(event)
        simulation_phases.append("Phase 3: No velocity")
        
    ut.set_neuron_tau1(model, 0, 1, (7, 255))
    ut.set_neuron_tau1(model, 0, 2, (7, 255))

    time.sleep(0.5)

    ut.set_neuron_tau1(model, 0, 1, (4, 50))
    ut.set_neuron_tau1(model, 0, 2, (4, 200))# save the drift for this cycle
    
    time.sleep(0.5)
    
    print(f"Simulation complete! Total events collected: {len(all_events)}")
    return all_events


def create_raster_plot(all_events, time_phase1, time_phase2, time_phase3, spike_pop_id):
    """
    Create raster plot from simulation events.
    
    Parameters:
    - all_events: List of spike events from simulation
    - time_phase1, time_phase2, time_phase3: Phase durations for marking transitions
    - spike_pop_id: Population that was stimulated (for title)
    
    Returns:
    - None (displays plot)
    """
    print("Creating raster plot...")
    
    if len(all_events) == 0:
        print("No events to plot!")
        return
    
    # Extract event data and normalize timestamps to start from 0
    raw_timestamps = [e.timestamp * 1e-6 for e in all_events]  # Convert to seconds
    start_time = min(raw_timestamps)  # Get the first timestamp
    event_times = [t - start_time for t in raw_timestamps]  # Normalize to start from 0
    event_neuron_ids = [e.neuron_id for e in all_events]
    
    # Create neuron ID to population mapping for coloring
    nid_to_pop = {}
    for pop_idx, pop in enumerate(ring_pops):
        for n in pop:
            nid_to_pop[n.neuron_id] = pop_idx
    
    # Give inhibitory population its own index
    INH_IDX = NBINS
    for n in pop_inhibitory:
        nid_to_pop[n.neuron_id] = INH_IDX
    
    # Map colors
    import matplotlib.cm as cm
    pop_colors = cm.get_cmap('tab10', NBINS+1)
    evt_colors = []
    for nid in event_neuron_ids:
        pop_idx = nid_to_pop.get(nid, None)
        if pop_idx is None or pop_idx == INH_IDX:
            evt_colors.append('lightgrey')
        else:
            evt_colors.append(pop_colors(pop_idx))
    
    # Create the plot
    plt.figure(figsize=(15, 8))
    plt.scatter(event_times, event_neuron_ids, s=3, c=evt_colors, alpha=0.7)
    
    # Add vertical lines to mark phase transitions
    plt.axvline(time_phase1, color='red', linestyle='--', linewidth=2, alpha=0.8, label='Velocity ON')
    plt.axvline(time_phase1 + time_phase2, color='blue', linestyle='--', linewidth=2, alpha=0.8, label='Velocity OFF') 
    
    # Add population labels and set y-axis limits
    y_max = max(event_neuron_ids) if event_neuron_ids else 100
    plt.ylim(0, y_max + 5)
    
    plt.xlabel('Time (s)', fontsize=12)
    plt.ylabel('Neuron ID', fontsize=12)
    plt.title(f'Offline Simulation: Population {spike_pop_id} Stimulation with Velocity Modulation\n' + 
              f'Phase 1 (0-{time_phase1}s): Initial | Phase 2 ({time_phase1}-{time_phase1+time_phase2}s): +Velocity | Phase 3 ({time_phase1+time_phase2}-{time_phase1+time_phase2+time_phase3}s): -Velocity', 
              fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    #plt.show()


def analyze_peak_tracking(all_events, time_phase1, time_phase2, time_phase3, timestep=0.5):
    """
    Perform peak tracking analysis on simulation events.
    
    Parameters:
    - all_events: List of spike events from simulation
    - time_phase1, time_phase2, time_phase3: Phase durations for marking transitions
    - timestep: Time window size for analysis (seconds)
    
    Returns:
    - peak_positions_degrees: Array of peak positions in degrees
    - peak_times: Array of time points for each peak
    """
    print("Starting peak tracking analysis...")
    
    #print('events : ', all_events)
    if len(all_events) == 0:
        print("No events to analyze!")
        return np.array([]), np.array([])
    
    # Extract and normalize event data
    raw_timestamps = [e.timestamp * 1e-6 for e in all_events]
    start_time = min(raw_timestamps)
    event_times = [t - start_time for t in raw_timestamps]
    event_neuron_ids = [e.neuron_id for e in all_events]
    
    # Create neuron ID to population mapping
    nid_to_pop = {}
    for pop_idx, pop in enumerate(ring_pops):
        for n in pop:
            nid_to_pop[n.neuron_id] = pop_idx
    
    # Analysis parameters
    total_duration = time_phase1 + time_phase2 + time_phase3
    time_bins = np.arange(0, total_duration + timestep, timestep)
    
    # Storage for results
    peak_positions = []
    peak_times = []
    all_firing_rates = []
    fit_quality = []
    
    print(f"Analyzing {len(time_bins)-1} time windows...")
    
    for i in range(len(time_bins) - 1):
        t_start = time_bins[i]
        t_end = time_bins[i + 1]
        
        # Find events in this time window
        window_events = []
        for j, event_time in enumerate(event_times):
            if t_start <= event_time < t_end:
                window_events.append((event_time, event_neuron_ids[j]))
        
        # Calculate firing rates for each population
        firing_rates = np.zeros(NBINS)
        for event_time, neuron_id in window_events:
            if neuron_id in nid_to_pop and nid_to_pop[neuron_id] < NBINS:  # Exclude inhibitory
                pop_idx = nid_to_pop[neuron_id]
                firing_rates[pop_idx] += 1
        
        # Convert to Hz (events per second)
        firing_rates = firing_rates / timestep
        all_firing_rates.append(firing_rates.copy())
        
        #print('firing_rates : ', firing_rates/timestep)
        
        # Fit circular Gaussian
        mu, sigma, A = fit_circular_gaussian(firing_rates)
        
        print("mu ", mu)
        
        
        peak_positions.append(mu)
        peak_times.append(t_start + timestep/2)  # Center of time window
            
        
    
    peak_positions = np.array(peak_positions)
    peak_times = np.array(peak_times)
    fit_quality = np.array(fit_quality)
    
    # Convert peak positions to degrees (360° / 10 bins = 36° per bin)
    peak_positions_degrees = peak_positions * 360.0 / NBINS
    
    print(f"Successfully tracked {len(peak_positions)} peaks out of {len(time_bins)-1} time windows")
    
    return peak_positions_degrees, peak_times


def plot_peak_tracking(peak_positions_degrees, peak_times, time_phase1, time_phase2, time_phase3):
    """
    Create peak tracking plot from analysis results.
    
    Parameters:
    - peak_positions_degrees: Array of peak positions in degrees
    - peak_times: Array of time points for each peak
    - time_phase1, time_phase2, time_phase3: Phase durations for marking transitions
    
    Returns:
    - None (displays plot)
    """
    if len(peak_positions_degrees) == 0:
        print("No peak data to plot!")
        return
    
    # Create peak tracking plot
    plt.figure(figsize=(12, 6))
    plt.plot(peak_times, peak_positions_degrees, 'ro', markersize=2, alpha=0.8, label='Peak Position')
    
    plt.axvline(time_phase1, color='red', linestyle='--', linewidth=2, alpha=0.8, label='Velocity ON')
    plt.axvline(time_phase1 + time_phase2, color='blue', linestyle='--', linewidth=2, alpha=0.8, label='Velocity OFF')
    
    plt.xlabel('Time (s)', fontsize=12)
    plt.ylabel('Peak Position (degrees)', fontsize=12)
    plt.title('Ring Attractor Peak Position Over Time\n(Population Activity Center Tracking)', fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=10)
    
    plt.ylim(0, 360)
    plt.yticks(np.arange(0, 361, 45), [f'{int(deg)}°' for deg in np.arange(0, 361, 45)])
    
    for deg in np.arange(0, 361, 90):
        plt.axhline(deg, color='gray', linestyle=':', alpha=0.2, linewidth=0.8)
    
    plt.tight_layout()
    plt.show()


# ===== EXAMPLE USAGE =====
print("Functions created successfully!")
print("\nExample usage:")
print("# Run simulation")
print("events = run_offline_simulation(2.0, 7.0, 3.0, 5)")
print("\n# Create raster plot")
print("create_raster_plot(events, 2.0, 7.0, 3.0, 5)")
print("\n# Analyze peak tracking")
print("peak_degrees, peak_times = analyze_peak_tracking(events, 2.0, 7.0, 3.0)")
print("\n# Plot peak tracking results")
print("plot_peak_tracking(peak_degrees, peak_times, 2.0, 7.0, 3.0)")

Functions created successfully!

Example usage:
# Run simulation
events = run_offline_simulation(2.0, 7.0, 3.0, 5)

# Create raster plot
create_raster_plot(events, 2.0, 7.0, 3.0, 5)

# Analyze peak tracking
peak_degrees, peak_times = analyze_peak_tracking(events, 2.0, 7.0, 3.0)

# Plot peak tracking results
plot_peak_tracking(peak_degrees, peak_times, 2.0, 7.0, 3.0)


In [24]:
nbins_test = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

peak_population_dic = {pop: [] for pop in range(NBINS)}

num_cycle=10
t1,t2,t3=2,7,3
timestep = 0.5

p_vel = 0.5

conn = 5

# Pre-compute a neuron-ID → pop-index lookup for speed
nid_to_pop = {n.neuron_id: pop_idx
              for pop_idx, pop in enumerate(ring_pops)
              for n in pop}

sink_node.get_events()  # Clear any old events
api.reset_timestamp()

for target_pop in nbins_test:
    

    peak_cycle_dic = {cycle: [] for cycle in range(num_cycle)}

    ut.set_neuron_tau1(model, 0, 1, (4, 50))
    ut.set_neuron_tau1(model, 0, 2, (4, 200))# save the drift for this cycle
    
    for j in range(num_cycle):
        print("Cycle number: ", j)
        events=run_offline_simulation(t1,t2,t3,target_pop)
        
        #create_raster_plot(events, t1, t2, t3, target_pop)
        
        peaks_pos, peaks_time = analyze_peak_tracking(events, t1, t2, t3, timestep) 
        peak_cycle_dic[j].append((peaks_pos, peaks_time))
           
    
   
    #all_pos = np.array([peak_cycle_dic[j][0][0] for j in range(len(peak_cycle_dic))])
    #print("all pos ", all_pos)
    
    # calculate mean & std at each timestep within peak_cycle_dic
    #mean_pos = np.mean(all_pos, axis=0)
    #std_pos = np.std(all_pos, axis=0)
    
    
    all_pos = np.array([peak_cycle_dic[j][0][0] for j in range(num_cycle)])
    print("all pos: ", all_pos)
    all_pos_rad = np.deg2rad(all_pos)
    print("all pos rad: ", all_pos_rad)

    mean_pos = np.rad2deg(np.angle(np.nanmean(np.exp(1j * all_pos_rad), axis=0)))
    print("mean_pos: ", mean_pos)

    std_pos = np.rad2deg(np.nanstd(np.angle(np.exp(1j * all_pos_rad)), axis=0))
    print("std_pos: ", std_pos)
    
    # Handle negative angles
    mean_pos = np.where(mean_pos < 0, mean_pos + 360, mean_pos)

    # append mean & std to peak_population_dic
    peak_population_dic[target_pop] = (mean_pos, std_pos, peaks_time)

# todo: plot 5x2 subplots. one scatter plot per population. each dot is mean_pos + error bar with value given by std_pos. take info from peak_population_dic
        
print(peak_cycle_dic)

plt.show()


Cycle number:  0

Starting offline simulation: Pop 0, phases: 2s, 7s, 3s
Phase 1: Stimulating population 0 for 2 seconds...
Phase 1 complete. Collected 56 events.
Phase 2: Adding velocity connections and running for 7 seconds...
Velocity connections added.
Phase 2 complete. Collected 295 events.
Phase 3: Removing velocity connections and running for 3 seconds...
Velocity connections removed.
Phase 3 complete. Collected 96 events.
Simulation complete! Total events collected: 447
Starting peak tracking analysis...
Analyzing 24 time windows...
mu  0.0
mu  0.0
mu  0.0
mu  9.0
mu  0.0
mu  1.1384196109248952
mu  1.382286624547932
mu  1.644369085917312
mu  1.8366234488940676
mu  2.8755753374139923
mu  4.943079671683657
mu  4.786887738341265
mu  4.925554804735794
mu  4.71341845297182
mu  5.064309489798189
mu  4.786887738341265
mu  4.962440753457691
mu  5.638139376207757
mu  5.924269293407344
mu  5.500000007389339
mu  5.311977007461331
mu  6.423584178591799
mu  5.058131874186015
mu  5.165479672

In [25]:
# save peak_population_dic for velocity one as .npy

number_connections = 'five_connections_per_pop'

np.save(f'peak_population_dic_velocity_connection_{number_connections}.npy', peak_population_dic)

# To load it back later:
# peak_population_dic = np.load('peak_population_dic_velocity.npy', allow_pickle=True).item()

In [26]:
import matplotlib.pyplot as plt

# Create 5x2 subplots
fig, axes = plt.subplots(2, 5, figsize=(20, 10))
axes = axes.flatten()  # Make it easier to index

for i, pop in enumerate(range(len(nbins_test))):
    if pop in peak_population_dic:
        mean_pos, std_pos, _ = peak_population_dic[pop]
        times = np.arange(len(mean_pos)) * timestep  # Create time array
        
        axes[i].errorbar(times, mean_pos, yerr=std_pos, marker='o', capsize=2, linestyle='none')
        axes[i].set_title(f'Population {pop}')
        axes[i].set_ylim(-10,370)
        if i >= len(nbins_test)/2:
           axes[i].set_xlabel('Time (s)')
        if i == 0 or i == len(nbins_test)/2:
            axes[i].set_ylabel('Peak position')
        
        axes[i].axvline(t1, ls='--', lw=0.8, c='red', alpha=0.4, label='Velocity ON')
        axes[i].axvline(t1+t2, ls='--', lw=0.8, c='blue', alpha=0.4, label='Velocity OFF')


plt.tight_layout()
plt.show()

In [27]:
import numpy as np
import matplotlib.pyplot as plt

# Calculate initial degrees for each population (36 degrees per population)
initial_degrees = {pop: pop * 36 for pop in range(10)}

# Adjust means by subtracting initial position and handle boundary conditions
adjusted_means = {}
for pop in peak_population_dic:
    if len(peak_population_dic[pop]) > 0:  # Check if data exists
        mean_pos = peak_population_dic[pop][0]  # First element (means)
        initial_deg = initial_degrees[pop]
        
        # Subtract initial position
        diff = mean_pos - initial_deg
        
        # Handle boundary conditions: wrap to [-180, 180] then to [0, 360]
        diff = ((diff + 180) % 360) - 180  # Wrap to [-180, 180]
        adjusted = np.where(diff < 0, diff + 360, diff)  # Convert negative to [180, 360]
        
        adjusted_means[pop] = adjusted

# Calculate mean and std across populations (ignoring NaNs)
if len(adjusted_means) > 0:
    # Stack all adjusted means
    all_adjusted = np.array(list(adjusted_means.values()))
    
    # Convert to radians for circular statistics
    all_adjusted_rad = np.deg2rad(all_adjusted)
    
    # Calculate circular mean and std across populations
    population_mean = np.rad2deg(np.angle(np.nanmean(np.exp(1j * all_adjusted_rad), axis=0)))
    population_std = np.rad2deg(np.nanstd(np.angle(np.exp(1j * all_adjusted_rad)), axis=0))
    
    # Handle negative angles
    population_mean = np.where(population_mean < 0, population_mean + 360, population_mean)
    
    # Create time array
    times = np.arange(len(population_mean)) * timestep
    
    # Plot
    plt.figure(figsize=(12, 8))
    
    # Plot individual populations (adjusted)
    for pop, adj_mean in adjusted_means.items():
        plt.plot(times, adj_mean, alpha=0.3, linewidth=1, label=f'Pop {pop} (adjusted)')
    
    # Plot population mean with error bars
    plt.errorbar(times, population_mean, yerr=population_std, 
                marker='o', linewidth=2, markersize=4, capsize=3, 
                color='black', label='Population Mean ± SD')
    
    # Add phase markers
    plt.axvline(t1, ls='--', lw=1.5, c='red', alpha=0.7, label='Velocity ON')
    plt.axvline(t1+t2, ls='--', lw=1.5, c='blue', alpha=0.7, label='Velocity OFF')
    
    plt.xlabel('Time (s)')
    plt.ylabel('Adjusted Peak Position (degrees)')
    plt.title('Ring Attractor: Population-Adjusted Peak Positions\n(Each population normalized to its initial position)')
    plt.ylim(0, 360)
    plt.grid(True, alpha=0.3)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
    
    print("Population statistics:")
    print(f"Mean drift: {np.nanmean(population_mean):.1f}°")
    print(f"Mean variability: {np.nanmean(population_std):.1f}°")
else:
    print("No valid data found in peak_population_dic")

Population statistics:
Mean drift: 97.8°
Mean variability: 29.8°


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Calculate initial degrees for each population (36 degrees per population)
initial_degrees = {pop: pop * 36 for pop in range(10)}

# Adjust means by subtracting initial position and handle boundary conditions
adjusted_means = {}
for pop in peak_population_dic:
    if len(peak_population_dic[pop]) > 0:  # Check if data exists
        mean_pos = peak_population_dic[pop][0]  # First element (means)
        initial_deg = initial_degrees[pop]
        
        # Subtract initial position
        diff = mean_pos - initial_deg
        
        # Handle boundary conditions: wrap to [-180, 180] and keep it there
        diff = ((diff + 180) % 360) - 180  # Wrap to [-180, 180]
        adjusted = diff  # Keep in [-180, 180] range
        
        adjusted_means[pop] = adjusted

# Calculate mean and std across populations (ignoring NaNs)
if len(adjusted_means) > 0:
    # Stack all adjusted means
    all_adjusted = np.array(list(adjusted_means.values()))
    
    # Convert to radians for circular statistics
    all_adjusted_rad = np.deg2rad(all_adjusted)
    
    # Calculate circular mean and std across populations
    population_mean = np.rad2deg(np.angle(np.nanmean(np.exp(1j * all_adjusted_rad), axis=0)))
    population_std = np.rad2deg(np.nanstd(np.angle(np.exp(1j * all_adjusted_rad)), axis=0))
    
    # Handle negative angles (keep in [-180, 180])
    # population_mean = np.where(population_mean < 0, population_mean + 360, population_mean)
    
    # Create time array
    times = np.arange(len(population_mean)) * timestep
    
    # Plot
    plt.figure(figsize=(12, 8))
    
    # Plot individual populations (adjusted)
    for pop, adj_mean in adjusted_means.items():
        plt.plot(times, adj_mean, alpha=0.3, linewidth=1, label=f'Pop {pop} (adjusted)')
    
    # Plot population mean with error bars
    plt.errorbar(times, population_mean, yerr=population_std, 
                marker='o', linewidth=2, markersize=4, capsize=3, 
                color='black', label='Population Mean ± SD')
    
    # Add phase markers
    plt.axvline(t1, ls='--', lw=1.5, c='red', alpha=0.7, label='Velocity ON')
    plt.axvline(t1+t2, ls='--', lw=1.5, c='blue', alpha=0.7, label='Velocity OFF')
    
    plt.xlabel('Time (s)')
    plt.ylabel('Adjusted Peak Position (degrees from start)')
    plt.title('Ring Attractor: Population-Adjusted Peak Positions\n(Each population normalized to its initial position)')
    plt.ylim(-180, 180)
    #plt.ylim(-100, 100)
    plt.grid(True, alpha=0.3)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
    
    print("Population statistics:")
    print(f"Mean drift: {np.nanmean(population_mean):.1f}°")
    print(f"Mean variability: {np.nanmean(population_std):.1f}°")
else:
    print("No valid data found in peak_population_dic")

Population statistics:
Mean drift: 52.8°
Mean variability: 29.8°


In [ ]:
# ===== LINEAR FIT ON POPULATION_MEAN IN TIME WINDOW =====

from scipy import stats
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

def linear_func(x, slope, intercept):
    """Linear function for fitting"""
    return slope * x + intercept

def perform_linear_fit_with_window(times, population_mean, population_std, t_start, t_end, use_weights=True):
    """
    Perform linear fit on population_mean within a specific time window
    
    Parameters:
    - times: array of time points
    - population_mean: array of population mean values
    - population_std: array of population std values
    - t_start: start time of fitting window
    - t_end: end time of fitting window
    - use_weights: whether to use std for weighted fitting
    
    Returns:
    - slope, intercept, r_squared, fit_times, fit_values
    """
    
    # Find indices within the time window
    window_mask = (times >= t_start) & (times <= t_end)
    
    if np.sum(window_mask) < 2:
        print(f"Warning: Only {np.sum(window_mask)} data points in window [{t_start}, {t_end}]")
        return None, None, None, None, None
    
    # Extract data within the window
    fit_times = times[window_mask]
    fit_mean = population_mean[window_mask]
    fit_std = population_std[window_mask]
    
    print(f"Fitting {len(fit_times)} data points in window [{t_start:.1f}, {t_end:.1f}] seconds")
    
    # Remove any NaN values
    valid_mask = ~np.isnan(fit_mean) & ~np.isnan(fit_std)
    fit_times = fit_times[valid_mask]
    fit_mean = fit_mean[valid_mask]
    fit_std = fit_std[valid_mask]
    
    if len(fit_times) < 2:
        print("Error: Not enough valid data points for fitting")
        return None, None, None, None, None
    
    try:
        if use_weights and np.any(fit_std > 0):
            # Use inverse of std as weights (avoid division by zero)
            weights = 1.0 / np.maximum(fit_std, np.min(fit_std[fit_std > 0]) * 0.1)
            
            # Weighted linear fit
            popt, pcov = curve_fit(linear_func, fit_times, fit_mean, sigma=1/weights)
            slope, intercept = popt
            
            print(f"Weighted fit used (based on std)")
        else:
            # Simple linear regression without weights
            slope, intercept, r_value, p_value, std_err = stats.linregress(fit_times, fit_mean)
            print(f"Unweighted fit used")
        
        # Calculate R-squared
        fit_values = linear_func(fit_times, slope, intercept)
        ss_res = np.sum((fit_mean - fit_values) ** 2)
        ss_tot = np.sum((fit_mean - np.mean(fit_mean)) ** 2)
        r_squared = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0
        
        print(f"Fit results:")
        print(f"  - Slope: {slope:.4f} degrees/second")
        print(f"  - Intercept: {intercept:.4f} degrees")
        print(f"  - R²: {r_squared:.4f}")
        
        return slope, intercept, r_squared, fit_times, fit_values
        
    except Exception as e:
        print(f"Error during fitting: {e}")
        return None, None, None, None, None

In [ ]:
# ===== LOAD .NPY DICTIONARY FILES =====

import os
import glob
import numpy as np




# Define the search path for .npy files in the JointAttractorNets folder
search_path = "/home/fferrari-iit.local/Code/Dynapse/JointAttractorNets/*.npy"

# Find all .npy files in the main folder (excluding subdirectories for now)
npy_files = glob.glob(search_path)

print(f"Found {len(npy_files)} .npy files:")
for file in npy_files:
    print(f"  - {os.path.basename(file)}")

# Dictionary to store loaded data (will be overwritten in each iteration as requested)
loaded_dict = {}

# Dictionary to store slopes with their corresponding number of connections
slope_vs_connections = {}

def extract_connection_number(filename):
    """
    Extract the number of connections from filename.
    Expected patterns:
    - 'peak_population_dic_velocity_connection_one_connection_per_pop.npy' -> 1
    - 'peak_population_dic_velocity_connection_two_connections_per_pop.npy' -> 2
    - 'peak_population_dic_velocity_connection_three_connections_per_pop.npy' -> 3
    - 'peak_population_dic_velocity_connection_four_connections_per_post_neur.npy' -> 4
    - 'peak_population_dic_velocity_connection_five_connections_per_pop.npy' -> 5
    """
    import re
    
    # Dictionary mapping word numbers to integers
    word_to_num = {
        'one': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5,
        'six': 6, 'seven': 7, 'eight': 8, 'nine': 9, 'ten': 10
    }
    
    # Try to find number words in the filename
    for word, num in word_to_num.items():
        if word in filename.lower():
            return num
    
    # If no word numbers found, try to find digit patterns
    # Look for patterns like "_1_", "_2_", etc.
    digit_match = re.search(r'_(\d+)_', filename)
    if digit_match:
        return int(digit_match.group(1))
    
    # If nothing found, return None
    print(f"Warning: Could not extract connection number from {filename}")
    return None

initial_degrees = {pop: pop * 36 for pop in range(10)}
t1,t2,t3=2,7,3
timestep = 0.5  # Define timestep for time calculation

# Load each .npy file in a for loop
for npy_file in npy_files:
    try:
        filename = os.path.basename(npy_file)
        print(f"\nLoading: {filename}")
        
        # Extract number of connections from filename
        num_connections = extract_connection_number(filename)
        print(f"  - Number of connections: {num_connections}")
        
        # Load the .npy file (assuming it contains a dictionary)
        loaded_dict = np.load(npy_file, allow_pickle=True).item()
        
        # Print some info about the loaded dictionary
        print(f"  - Type: {type(loaded_dict)}")
        if isinstance(loaded_dict, dict):
            print(f"  - Keys: {list(loaded_dict.keys())}")
            print(f"  - Number of keys: {len(loaded_dict)}")
            
        adjusted_means = {}
        for pop in loaded_dict:
            if len(loaded_dict[pop]) > 0:  
                mean_pos = loaded_dict[pop][0]  
                initial_deg = initial_degrees[pop]
                
                diff = mean_pos - initial_deg
                
                diff = ((diff + 180) % 360) - 180  
                adjusted = diff 
                
                adjusted_means[pop] = adjusted

        if len(adjusted_means) > 0:
            all_adjusted = np.array(list(adjusted_means.values()))
            
            all_adjusted_rad = np.deg2rad(all_adjusted)
            

            population_mean = np.rad2deg(np.angle(np.nanmean(np.exp(1j * all_adjusted_rad), axis=0)))
            population_std = np.rad2deg(np.nanstd(np.angle(np.exp(1j * all_adjusted_rad)), axis=0))
            times = np.arange(len(population_mean)) * timestep
            
            fit_start = t1  # Start of velocity phase
            fit_end = t1 + t2  # End of velocity phase
            
            print(f"Performing linear fit in time window: [{fit_start}, {fit_end}] seconds")
            print(f"This corresponds to the velocity phase of the simulation")
            
            # Perform the fit
            slope, intercept, r_squared, fit_times, fit_values = perform_linear_fit_with_window(
                times, population_mean, population_std, fit_start, fit_end, use_weights=True
            )
            
            if slope is not None and num_connections is not None:
                print(f"  - Slope: {slope:.4f} degrees/second")
                print(f"  - Intercept: {intercept:.4f} degrees")
                print(f"  - R²: {r_squared:.4f}")
                
                # Store slope with number of connections
                slope_vs_connections[num_connections] = {
                    'slope': slope,
                    'intercept': intercept,
                    'r_squared': r_squared,
                    'filename': filename,
                    'velocity_deg_per_sec': slope
                }
                print(f"  - Stored result for {num_connections} connections")
            else:
                print(f"  - Fitting failed or connection number not extracted")

    except Exception as e:
        print(f"  - Error loading {filename}: {e}")

print(f"\nLast loaded dictionary keys: {list(loaded_dict.keys()) if isinstance(loaded_dict, dict) else 'Not a dictionary'}")

# ===== SUMMARY OF RESULTS =====
print("\n" + "="*50)
print("SUMMARY: Slope vs Number of Connections")
print("="*50)

if slope_vs_connections:
    print(f"{'Connections':<12} {'Slope (°/s)':<12} {'R²':<8} {'Filename'}")
    print("-" * 60)
    
    # Sort by number of connections
    for num_conn in sorted(slope_vs_connections.keys()):
        result = slope_vs_connections[num_conn]
        print(f"{num_conn:<12} {result['slope']:<12.4f} {result['r_squared']:<8.4f} {result['filename']}")
    
    # Create arrays for potential plotting
    connections = np.array(sorted(slope_vs_connections.keys()))
    slopes = np.array([slope_vs_connections[n]['slope'] for n in connections])
    r_squares = np.array([slope_vs_connections[n]['r_squared'] for n in connections])
    
    print(f"\nArrays created for analysis:")
    print(f"connections = {connections}")
    print(f"slopes = {slopes}")
    print(f"r_squares = {r_squares}")
    
else:
    print("No successful fits found!")

print(f"\nAll results stored in 'slope_vs_connections' dictionary")

Found 5 .npy files:
  - peak_population_dic_velocity_connection_four_connections_per_post_neur.npy
  - peak_population_dic_velocity_connection_one_connection_per_pop.npy
  - peak_population_dic_velocity_connection_two_connections_per_pop.npy
  - peak_population_dic_velocity_connection_three_connections_per_pop.npy
  - peak_population_dic_velocity_connection_five_connections_per_pop.npy

Loading: peak_population_dic_velocity_connection_four_connections_per_post_neur.npy
  - Type: <class 'dict'>
  - Keys: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
  - Number of keys: 10
Performing linear fit in time window: [2, 9] seconds
This corresponds to the velocity phase of the simulation
Fitting 15 data points in window [2.0, 9.0] seconds
Weighted fit used (based on std)
Fit results:
  - Slope: 12.3630 degrees/second
  - Intercept: -23.2559 degrees
  - R²: 0.9859
  - Slope: 12.3630 degrees/second
  - Intercept: -23.2559 degrees
  - R²: 0.9859

Loading: peak_population_dic_velocity_connection_one_connection_pe